In [ ]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    
    evidence_per_prototype: int = 2         
    topk_feature: int = 3                   
    probe_C: float = 1.0                    
    calib_n_splits: int = 5                 

    # --- evidence sentence retrieval: BM25 + Cosine + Classifier-Based Relevance ---
    top_m_evidence: int = 3             # Top-m evidence sentences per predicted IPC
    evidence_bm25_weight: float = 0.30
    evidence_cosine_weight: float = 0.40
    evidence_classifier_weight: float = 0.30
    relevance_clf_hidden: int = 128
    relevance_clf_epochs: int = 5
    relevance_clf_lr: float = 1e-3
    relevance_clf_batch_size: int = 64
    relevance_clf_neg_per_pos: int = 4   

    # explanation generator (LLM-Based Reasoning Generation, with CoT prompting)
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 220

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
#  Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")



_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation, gold_sections=None):
    """(case sentence, positive prototype code) pairs, built from the per-sentence
    `explanation` field (exact match first, fuzzy fallback), with a fallback to
    pairing every sentence with every gold label when the explanation dict yields
    nothing for a case that DOES have gold labels (same logic as Run 1). No
    longer just "kept for parity" -- this now feeds train_relevance_classifier()
    below, which needs (sentence, code) supervision for the evidence retrieval
    classifier."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    if not pairs and gold_sections:
        for code in gold_sections:
            if code in prototype_texts:
                for s in sentences:
                    pairs.append((s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
#  Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------

observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# -------------------------------
# InLegalBERT prototype encoder 
# --------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))



encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)



_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------
#  Statute Prototype Bank
# --------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# ---------------------------------------------
#  Scoring, supervised probe, and prediction
# ---------------------------------------------
def compute_features(fact_text):
    """Returns (feature_vector (3*n_candidates,), evidence {code: [sentences]},
    display_scores {code: max sim}). Scores/features only over `candidate_codes`
    (observed labels) -- see section 5b. Three features per candidate:
      1. max      -- the single best-matching sentence's similarity (noisy:
                      one spurious sentence match can dominate a document).
      2. topk_mean -- mean of the top-k matching sentences' similarity, a
                      steadier version of (1).
      3. rank_feat -- a per-document softmax over the `max` scores across all
                      candidates, i.e. how much THIS candidate stands out
                      RELATIVE to the other candidates for this specific case.
                      Pure magnitude features let a class with systematically
                      higher raw similarity (e.g. the majority class) dominate
                      regardless of case; this relative feature gives the probe
                      a comparison signal that isn't just absolute magnitude,
                      which matters most for the lower-support classes.
    `evidence` here is the legacy top-2-by-max-similarity sentence list, kept
    for cheap diagnostics; final predictions use retrieve_evidence_sentences()
    in section 8b instead (BM25 + cosine + classifier, not max-sim alone).
    """
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    k = min(cfg.topk_feature, sim_matrix.shape[0])
    max_sim = sim_matrix.max(axis=0)
    topk_mean = np.sort(sim_matrix, axis=0)[-k:].mean(axis=0)
    rank_feat = np.exp(max_sim - max_sim.max())
    rank_feat = rank_feat / rank_feat.sum()
    features = np.concatenate([max_sim, topk_mean, rank_feat]).astype(np.float32)
    display_scores = {c: float(max_sim[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return features, evidence, display_scores



from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, 2*n_candidates) feature matrix;
    Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        feats, _, _ = compute_features(d["fact"])
        X.append(feats)
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Cross-validated (threshold, C) calibration -- SIXTH FIX below -- then
    final per-class probes refit on the full train+val pool."""
    from skmultilearn.model_selection import IterativeStratification

    pool = train_split + val_split
    X_pool, Y_by_code = build_feature_matrix(pool)
    Y_matrix = np.stack([Y_by_code[c] for c in candidate_codes], axis=1)

    kfold = IterativeStratification(n_splits=cfg.calib_n_splits, order=1)
    fold_indices = list(kfold.split(X_pool, Y_matrix))
    proba_grid = np.arange(0.05, 0.96, 0.01)

    
    C_GRID = (0.05, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0)

    thresholds, chosen_C = {}, {}
    for c in candidate_codes:
        y_full = np.array(Y_by_code[c])
        best = {"C": cfg.probe_C, "threshold": 0.5, "f1": -1.0}
        for C_val in C_GRID:
            oof = np.zeros(len(pool))
            for train_idx, hold_idx in fold_indices:
                y_tr = y_full[train_idx]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    # Degenerate fold for this class (no positives, or all
                    # positives) -- LogisticRegression can't fit; fall back to
                    # the fold's training prevalence as a constant probability.
                    oof[hold_idx] = y_tr.mean() if len(y_tr) else 0.0
                    continue
                clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C_val)
                clf.fit(X_pool[train_idx], y_tr)
                oof[hold_idx] = clf.predict_proba(X_pool[hold_idx])[:, 1]

            best_t, best_f1 = 0.5, -1.0
            for t in proba_grid:
                pred = (oof >= t).astype(int)
                f1 = f1_score(y_full, pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            if best_f1 > best["f1"]:
                best = {"C": C_val, "threshold": best_t, "f1": best_f1}

        thresholds[c] = best["threshold"]
        chosen_C[c] = best["C"]
        print(f"  [{c}] best C={best['C']:g}  {cfg.calib_n_splits}-fold OOF F1={best['f1']:.4f}  "
              f"@ proba-threshold={best['threshold']:.2f}  "
              f"(positives={int(y_full.sum())}/{len(y_full)} across train+val)")

    
    probes = {}
    for c in candidate_codes:
        y = np.array(Y_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=chosen_C[c])
        clf.fit(X_pool, y)
        probes[c] = clf

    return probes, thresholds


# --------------------------------------------------------------------------
# 8b. Evidence Sentence Retrieval: BM25 + Cosine Similarity +
#     Classifier-Based Relevance -> Top-m Evidence Sentences.
# --------------------------------------------------------------------------
def _bm25_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def bm25_sentence_scores(sentences, query_text):
    """BM25 lexical relevance of each of THIS case's sentences against a
    statute's title+description. The BM25 index is built per-case (over
    just that case's own sentences) so the score reflects which of the
    case's sentences best match the statute, not corpus-wide term rarity."""
    if not sentences:
        return np.zeros(0)
    bm25 = BM25Okapi([_bm25_tokenize(s) for s in sentences])
    scores = np.array(bm25.get_scores(_bm25_tokenize(query_text)), dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span > 1e-9 else np.zeros_like(scores)


def _minmax(x):
    x = np.asarray(x, dtype=np.float64)
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 1e-9 else np.zeros_like(x)


class RelevanceClassifier(nn.Module):
    """Classifier-Based Relevance head: P(sentence supports statute) from
    [sent_emb ; proto_emb ; sent_emb * proto_emb] using the same frozen,
    whitened InLegalBERT embeddings already computed for the similarity
    matrix. This classifier is a small trainable MLP -- the underlying
    encoder itself remains frozen, so this stays consistent with Run 2's
    "no contrastive fine-tuning" baseline."""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 3, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        )

    def forward(self, sent_emb, proto_emb):
        feat = torch.cat([sent_emb, proto_emb, sent_emb * proto_emb], dim=-1)
        return self.net(feat).squeeze(-1)


def train_relevance_classifier():
    """Trains RelevanceClassifier on (sentence, statute) pairs from the same
    `explanation`-derived positives as positive_pairs_from_explanation(),
    restricted to `candidate_codes`, with `relevance_clf_neg_per_pos` sampled
    negative candidate codes per positive. Frozen, whitened embeddings from
    the encoder are used as features, so this trains fast (a few epochs of a
    small MLP) and never touches the InLegalBERT weights."""
    pos_pairs = []
    for rec in train_split:
        pos_pairs.extend(positive_pairs_from_explanation(
            rec["fact"], rec.get("explanation", {}) or {}, rec["gold_sections"]))
    pos_pairs = [(s, c) for s, c in pos_pairs if c in candidate_codes]
    if not pos_pairs:
        print("[relevance-classifier] no (sentence, statute) positives found -- "
              "evidence retrieval will fall back to BM25 + cosine only.")
        return None

    code_to_idx = {c: i for i, c in enumerate(candidate_codes)}
    uniq_sentences = list({s for s, _ in pos_pairs})
    sent_emb_lookup = {s: e for s, e in zip(uniq_sentences, embed_and_whiten(uniq_sentences, cfg.max_length))}

    X_sent, X_proto, y = [], [], []
    for s, c in pos_pairs:
        X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[c]]); y.append(1.0)
        neg_pool = [cc for cc in candidate_codes if cc != c]
        for nc in random.sample(neg_pool, min(cfg.relevance_clf_neg_per_pos, len(neg_pool))):
            X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[nc]]); y.append(0.0)

    X_sent = torch.stack(X_sent).to(DEVICE)
    X_proto = torch.stack(X_proto).to(DEVICE)
    y = torch.tensor(y, dtype=torch.float32, device=DEVICE)

    clf = RelevanceClassifier(X_sent.shape[-1], cfg.relevance_clf_hidden).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=cfg.relevance_clf_lr)

    clf.train()
    for epoch in range(1, cfg.relevance_clf_epochs + 1):
        idx = torch.randperm(len(y), device=DEVICE)
        total, steps = 0.0, 0
        for i in range(0, len(y), cfg.relevance_clf_batch_size):
            b = idx[i:i + cfg.relevance_clf_batch_size]
            opt.zero_grad()
            logits = clf(X_sent[b], X_proto[b])
            loss = F.binary_cross_entropy_with_logits(logits, y[b])
            loss.backward()
            opt.step()
            total += loss.item(); steps += 1
        print(f"  [relevance-classifier] epoch {epoch}/{cfg.relevance_clf_epochs} "
              f"BCE loss = {total / max(1, steps):.4f}")
    clf.eval()
    return clf


@torch.no_grad()
def classifier_relevance_scores(clf, sentences, proto_vec):
    if clf is None or not sentences:
        return np.zeros(len(sentences))
    sent_emb = embed_and_whiten(sentences, cfg.max_length).to(DEVICE)
    proto_batch = proto_vec.to(DEVICE).unsqueeze(0).expand(len(sentences), -1)
    probs = torch.sigmoid(clf(sent_emb, proto_batch)).cpu().numpy()
    return probs


def retrieve_evidence_sentences(sentences, section_code, relevance_clf, top_m=None):
    """Score S = {S1..Sn} via BM25 + Cosine Similarity + Classifier-Based
    Relevance, min-max normalise each signal, combine with the configured
    weights, and return the Top-m evidence sentences (sentence, combined_score)
    for `section_code`, ranked highest first."""
    top_m = top_m or cfg.top_m_evidence
    if not sentences:
        return []

    proto_idx = candidate_codes.index(section_code)
    proto_vec = bank_emb[proto_idx]

    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    cosine_raw = (sent_emb @ proto_vec.unsqueeze(-1)).squeeze(-1).numpy()
    cosine = _minmax(cosine_raw)

    query_text = f"{prototype_titles.get(section_code, '')} {prototype_texts.get(section_code, '')}".strip()
    bm25 = bm25_sentence_scores(sentences, query_text)

    clf_scores = _minmax(classifier_relevance_scores(relevance_clf, sentences, proto_vec))

    combined = (cfg.evidence_bm25_weight * bm25
                + cfg.evidence_cosine_weight * cosine
                + cfg.evidence_classifier_weight * clf_scores)

    order = np.argsort(-combined)[:min(top_m, len(sentences))]
    return [(sentences[i], float(combined[i])) for i in order]


def predict_case(fact_text, probes, thresholds, relevance_clf=None):
    feats, _legacy_evidence, _ = compute_features(fact_text)
    feat = feats.reshape(1, -1)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]

    sentences = split_sentences(fact_text, cfg.max_sentences)
    results = []
    for c, p in chosen:
        evidence = retrieve_evidence_sentences(sentences, c, relevance_clf, cfg.top_m_evidence)
        results.append({
            "section": c,
            "score": round(float(p), 4),
            "evidence_sentences": [sent for sent, _ in evidence],
            "evidence_scores": [round(sc, 4) for _, sc in evidence],
        })
    return results


# --------------------------------------------------------------------------
# 9. LLM-Based Reasoning Generation: (IPC Section, Selected Evidence
#    Sentences, CoT prompting) -> Qwen -> Explanation.
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def build_cot_prompt(section_code, evidence_sentences, score):
    """Chain-of-Thought prompt: statute + ranked evidence -> (1) identify
    satisfied elements -> (2) connect each evidence sentence -> (3) final
    explanation. Matches Figure 1's "Input to LLM (IPC Section, Selected
    Evidence Sentences, CoT prompting)" box."""
    title = prototype_titles.get(section_code, "")
    statute_text = prototype_texts.get(section_code, "")
    evidence_block = "\n".join(f"  {i + 1}. {sent} (relevance={sc:.3f})"
                                for i, (sent, sc) in enumerate(evidence_sentences)) or "  (no evidence retrieved)"
    return (
        "You are a legal reasoning assistant analysing an Indian Penal Code (IPC) "
        "statute prediction. Think step by step before answering.\n\n"
        f"Candidate section: {section_code}{f' ({title})' if title else ''}\n"
        f"Statute text: {statute_text}\n"
        f"Probe confidence score: {score:.3f}\n\n"
        f"Evidence sentences retrieved from the case facts (ranked by combined "
        f"BM25 + cosine similarity + classifier relevance):\n{evidence_block}\n\n"
        "Step 1 - List which elements of the statute the evidence appears to satisfy.\n"
        "Step 2 - For each evidence sentence, state briefly how it supports (or "
        "weakens) applicability of this section.\n"
        "Step 3 - Give a final, concise legal explanation (2-4 sentences) linking "
        "the evidence to the statute.\n\n"
        "Respond with your Step 1 and Step 2 reasoning first, then a line "
        "'Final Explanation:' followed by the Step 3 explanation."
    )


def generate_explanation(section_code, evidence_sentences, score):
    """evidence_sentences: list of (sentence_text, combined_relevance_score) tuples,
    already ranked by retrieve_evidence_sentences(). Uses Qwen with CoT prompting
    when cfg.use_qwen_explainer=True; otherwise a deterministic template reproduces
    the same three-step structure without a decoder LLM."""
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        prompt = build_cot_prompt(section_code, evidence_sentences, score)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": prompt},
        ]
        chat_prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(chat_prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        full_output = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if "Final Explanation:" in full_output:
            return full_output.split("Final Explanation:", 1)[1].strip()
        return full_output

    # Deterministic fallback: same CoT structure, no decoder LLM required.
    title = prototype_titles.get(section_code, "")
    title_part = f" ({title})" if title else ""
    statute_text = prototype_texts.get(section_code, "")
    if not evidence_sentences:
        return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}), "
                f"but no evidence sentences were retrieved for this case.")
    top_sent, top_score = evidence_sentences[0]
    others = "; ".join(s[:100] for s, _ in evidence_sentences[1:])
    return (
        f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
        f"Step 1: the case facts describe conduct matching the elements of \"{statute_text[:160]}...\". "
        f"Step 2: the strongest supporting sentence (relevance {top_score:.3f}) is \"{top_sent[:160]}\""
        + (f", further supported by: {others[:200]}." if others else ".")
        + " Step 3 (final explanation): the retrieved evidence, ranked by BM25 + embedding "
          "similarity + classifier relevance, is consistent with this statute applying to the case."
    )


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, relevance_clf=None, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds, relevance_clf)
        for p in preds:
            evidence_pairs = list(zip(p["evidence_sentences"], p["evidence_scores"]))
            p["explanation"] = generate_explanation(p["section"], evidence_pairs, p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":

    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()

    print("\n[relevance classifier] training BM25+cosine+classifier evidence scorer "
          "on the frozen, whitened InLegalBERT embeddings ...")
    relevance_clf = train_relevance_classifier()

    run2_metrics, run2_predictions = evaluate_run(
        "run2_prototype_baseline", probes, thresholds, relevance_clf)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, topk_feature=3, probe_C=1.0, calib_n_splits=5, top_m_evidence=3, evidence_bm25_weight=0.3, evidence_cosine_weight=0.4, evidence_classifier_weight=0.3, relevance_clf_hidden=128, relevance_clf_epochs=5, relevance_clf_lr=0.001, relevance_clf_batch_size=64, relevance_clf_neg_per_pos=4, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=220, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female h

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] training + calibrating per-class probes:


Featurizing:   0%|          | 0/422 [00:00<?, ?it/s]

  [IPC 147] best C=30  5-fold OOF F1=0.3795  @ proba-threshold=0.54  (positives=62/422 across train+val)
  [IPC 201] best C=30  5-fold OOF F1=0.2357  @ proba-threshold=0.47  (positives=41/422 across train+val)
  [IPC 302] best C=100  5-fold OOF F1=0.5732  @ proba-threshold=0.34  (positives=145/422 across train+val)
  [IPC 376] best C=30  5-fold OOF F1=0.4179  @ proba-threshold=0.56  (positives=66/422 across train+val)
  [IPC 420] best C=100  5-fold OOF F1=0.5538  @ proba-threshold=0.65  (positives=64/422 across train+val)
  [IPC 498A] best C=100  5-fold OOF F1=0.6338  @ proba-threshold=0.66  (positives=67/422 across train+val)
  [IPC 506] best C=100  5-fold OOF F1=0.2597  @ proba-threshold=0.55  (positives=52/422 across train+val)

[relevance classifier] training BM25+cosine+classifier evidence scorer on the frozen, whitened InLegalBERT embeddings ...
  [relevance-classifier] epoch 1/5 BCE loss = 0.4857
  [relevance-classifier] epoch 2/5 BCE loss = 0.4268
  [relevance-classifier] epoch

[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.4143
              micro_f1: 0.4092
           weighted_f1: 0.4482
       macro_precision: 0.3402
          macro_recall: 0.6044
       micro_precision: 0.2996
          micro_recall: 0.6452
  exact_match_accuracy: 0.0777
          hamming_loss: 0.3204
